## 1. Environment Setup
We begin by installing the necessary libraries for web scraping (`requests`, `beautifulsoup4`) and data processing/AI tasks (`openai`, `pandas`).

# Habitaclia Scraper
Scrapes 100 listings from habitaclia.com using an LLM to parse each detail page.

In [46]:
# ── 1. INSTALL DEPENDENCIES ──────────────────────────────────────────────────
!pip install -q requests beautifulsoup4 openai

## 2. Configuration & Initialization
In this section, we define the API keys, base URLs, and constants used throughout the scraper. We also initialize the OpenAI client.

In [ ]:
import time, json, re
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────────
OPENAI_API_KEY = ""
BASE_URL          = "https://english.habitaclia.com"
# Changed template to allow different categories (homes vs rent)
LIST_URL_TEMPLATE = BASE_URL + "/{category}-barcelona{suffix}.htm"
OUTPUT_FILE = "habitaclia_listings.json"
TARGET_PER_CATEGORY = 50
DELAY_SECONDS     = 1.5
MODEL             = "gpt-5.4"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9",
}

client = OpenAI(api_key=OPENAI_API_KEY)
print("Config ready ✓")

Config ready ✓


## 3. Data Extraction Schema
To ensure consistent data extraction, we define a JSON schema that the LLM must follow when parsing HTML content into structured real-estate data.

In [48]:
# ── 3. SCHEMA (shared with the LLM) ──────────────────────────────────────────
SCHEMA = """
{
  "external_id":           "string or null",
  "title":                 "string",
  "transaction_type":      "buy | rent | null",
  "listing_type":          "flat | penthouse | house | duplex | room | null",
  "price":                 "integer or null",
  "surface_m2":            "integer or null",
  "rooms":                 "integer or null",
  "bathrooms":             "integer or null",
  "floor":                 "integer or null",
  "neighborhood":          "string or null",
  "street":                "string or null",
  "city":                  "string or null",
  "has_elevator":          "boolean or null",
  "has_garage":            "boolean or null",
  "has_furniture":         "boolean or null",
  "has_terrace":           "boolean or null",
  "has_air_conditioning":  "boolean or null",
  "has_heating":           "boolean or null",
  "description":           "string or null",
  "year_built":            "integer or null"
}
"""

SYSTEM_PROMPT = (
    "You are a real-estate data extraction assistant. "
    "Given raw HTML from a habitaclia.com listing page, extract the fields "
    "described below and return ONLY a valid JSON object — no markdown, no "
    "explanation, no extra keys.\n\n"
    "Return exactly one JSON object with all schema keys in the order shown below. If a field is missing, ambiguous, or not supported by the HTML, use null rather than guessing. Before finalizing, verify that the output is valid JSON, contains no extra keys, and every value matches the requested type.\n"
    "All user-facing text fields should be translated to English.\n"
    "If no title is provided, write a short attractive title which makes the property stand out.\n"
    "Tailor the description so that it is in English, polished, clear, and contains details of the listing. Maintain information from the provided description if available, but do not invent facts not supported by the HTML..\n\n"
    "Schema:\n" + SCHEMA
)
print("Schema defined ✓")
print(SYSTEM_PROMPT)

Schema defined ✓
You are a real-estate data extraction assistant. Given raw HTML from a habitaclia.com listing page, extract the fields described below and return ONLY a valid JSON object — no markdown, no explanation, no extra keys.

Return exactly one JSON object with all schema keys in the order shown below. If a field is missing, ambiguous, or not supported by the HTML, use null rather than guessing. Before finalizing, verify that the output is valid JSON, contains no extra keys, and every value matches the requested type.
All user-facing text fields should be translated to English.
If no title is provided, write a short attractive title which makes the property stand out.
Tailor the description so that it is in English, polished, clear, and contains details of the listing. Maintain information from the provided description if available, but do not invent facts not supported by the HTML..

Schema:

{
  "external_id":           "string or null",
  "title":                 "string",


## 4. Helper Functions
We define modular functions to handle network requests, HTML cleaning, and link extraction. This keeps our main scraping loop clean and easy to follow.

In [49]:
from bs4 import BeautifulSoup
import re

def fetch(url: str) -> str | None:
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  [fetch error] {url} → {e}")
        return None

def extract_relevant_sections(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, "html.parser")
    result = BeautifulSoup("", "html.parser")
    container = soup.select_one("main div.content-detail-filter")
    if not container: return ""
    for t in container.select("script, style, svg, noscript, iframe"): t.decompose()
    for img in container.select("img"): img.decompose()
    aside_to_remove = container.select_one("aside#js-similar-top.similar")
    if aside_to_remove: aside_to_remove.decompose()
    sections = container.select("section.detail, section.summary")
    allowed_tags = {"p", "ul", "ol", "li", "h1", "h2", "h3", "h4", "strong", "em", "a"}
    for section in sections:
        for tag in section.find_all(True):
            if tag.name not in allowed_tags: tag.unwrap()
            else:
                if tag.name != "a": tag.attrs = {}
        result.append(section)
    text = str(result)
    text = re.sub(r"\n\s*\n", "\n", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text[:80_000]

def extract_image_urls(raw_html: str) -> list[str]:
    soup = BeautifulSoup(raw_html, "html.parser")
    urls = []
    for img in soup.select("div.flex-images img"):
        src = img.get("src")
        if src:
            if src.startswith("//"): src = "https:" + src
            urls.append(src)
    return list(set(urls))

def extract_listing_links(html: str) -> list:
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for article in soup.select("article.js-list-item[data-href]"):
        href = article["data-href"].split("?")[0]
        if href.startswith("//"): href = "https:" + href
        if href.startswith("http") and href not in links: links.append(href)
    return links

def listing_url_for_page(category: str, page: int) -> str:
    suffix = "" if page == 1 else f"-{page - 1}"
    return LIST_URL_TEMPLATE.format(category=category, suffix=suffix)

def parse_with_llm(slimmed_html: str, listing_url: str) -> dict | None:
    user_msg = f"Listing URL: {listing_url}\n\nHTML:\n{slimmed_html}"
    try:
        response = client.responses.create(model=MODEL, instructions=SYSTEM_PROMPT, input=user_msg)
        raw = response.output_text.strip()
        if "```json" in raw: raw = raw.split("```json")[1].split("```")[0].strip()
        return json.loads(raw)
    except Exception as e:
        print(f"  [llm error]  {e}")
        return None

## 5. Main Scraping Loop
This loop iterates through the listing pages for both 'homes' and 'rent' categories, fetches individual listing details, and uses the LLM to parse them.

In [50]:
results:   list = []
seen_urls: set  = set()

for category in ["homes", "rent"]:
    category_count = 0
    page = 1
    print(f"\n⌒ Starting category: {category}")

    while category_count < TARGET_PER_CATEGORY:
        list_url = listing_url_for_page(category, page)
        print(f"\n│ Page {page}: {list_url}")

        list_html = fetch(list_url)
        if not list_html: break

        detail_links = extract_listing_links(list_html)
        if not detail_links: break

        for url in detail_links:
            if category_count >= TARGET_PER_CATEGORY: break
            if url in seen_urls: continue
            seen_urls.add(url)

            print(f"  ↓ [{category_count+1}/{TARGET_PER_CATEGORY}] {url}")
            detail_html = fetch(url)
            time.sleep(DELAY_SECONDS)

            if not detail_html: continue
            relevant_html = extract_relevant_sections(detail_html)

            if relevant_html:
                listing = parse_with_llm(relevant_html, url)
                if listing:
                    listing["_source_url"] = url
                    listing["image_urls"] = extract_image_urls(detail_html)
                    results.append(listing)
                    category_count += 1
                    print(f"     ✓ {listing.get('title', '')[:50]}...")
                time.sleep(DELAY_SECONDS)
        page += 1

print(f"\n⌑ Total collected: {len(results)} listings.")


⌒ Starting category: homes

│ Page 1: https://english.habitaclia.com/homes-barcelona.htm
  ↓ [1/50] https://english.habitaclia.com/buy-flat-la_nova_esquerra_de_l_eixample-barcelona-i27066000000664.htm
     ✓ Flat in La Nova Esquerra de l'Eixample, Barcelona...
  ↓ [2/50] https://english.habitaclia.com/buy-flat-dreta_de_l_eixample-barcelona-i8166003785996.htm
     ✓ Flat with heating in Dreta de l'Eixample Barcelona...
  ↓ [3/50] https://english.habitaclia.com/buy-penthouse-l_antiga_esquerra_de_l_eixample-barcelona-i8166003796609.htm
     ✓ Penthouse with heating in L’Antiga Esquerra de l’E...
  ↓ [4/50] https://english.habitaclia.com/buy-flat-sagrada_familia-barcelona-i27066000000663.htm
     ✓ Renovated 4-bedroom flat near Sagrada Família...
  ↓ [5/50] https://english.habitaclia.com/buy-flat-sagrada_familia-barcelona-i8166003809623.htm
     ✓ Flat with heating in Sagrada Família Barcelona...
  ↓ [6/50] https://english.habitaclia.com/buy-flat-dreta_de_l_eixample-barcelona-i49750043382

## 6. Data Validation & Quality Scoring
After collection, we filter out incomplete records and calculate a 'Quality Score' based on the presence of critical information and description depth.

In [82]:
def calculate_quality_score(item):
    score = 10

    # 1. Critical Fields (-2 each)
    critical_fields = ['price', 'surface_m2', 'rooms', 'bathrooms', 'neighborhood', 'description', 'street']
    for field in critical_fields:
        if item.get(field) is None:
            score -= 2

    # 2. Non-Critical Fields (-0.5 each)
    non_critical = ['floor', 'year_built', 'has_elevator', 'has_garage', 'has_heating']
    for field in non_critical:
        if item.get(field) is None:
            score -= 0.5

    # 3. Content Penalties
    desc = item.get('description') or ''
    desc_len = len(desc)
    if desc_len < 200:
        score -= 4
    elif desc_len < 500:
        score -= 2
    elif desc_len < 1000:
        score -= 1

    images = item.get('image_urls') or []
    img_count = len(images)
    if img_count < 10:
        score -= 2
    elif img_count < 20:
        score -= 1

    score = min(10, max(0, score))
    return int(round(score))

# Filter results: drop items where critical fields are None and show them
required_fields = ['transaction_type', 'listing_type', 'price', 'city']
filtered_results = []
dropped_count = 0

print("--- Dropped Listings Report ---")
for item in results:
    missing = [f for f in required_fields if item.get(f) is None]
    if missing:
        print(f"Dropped: {item.get('title', 'No Title')} | URL: {item.get('_source_url', 'N/A')}")
        print(f"  № Missing fields: {missing}")
    else:
        filtered_results.append(item)

results = filtered_results
print("-------------------------------\n")

# Apply to remaining results
for listing in results:
    listing['quality_score'] = calculate_quality_score(listing)

# Update DataFrame
df = pd.DataFrame(results)

print(f"Total listings after filtering: {len(df)}")
print(f"Median Quality Score: {df['quality_score'].median()}")
print("\nListings count per score:")
print(df['quality_score'].value_counts().sort_index(ascending=False))

print("\nBottom 10 listings by score:")
display(df[['title', 'price', 'quality_score']].sort_values(by='quality_score', ascending=True).head(10))

--- Dropped Listings Report ---
-------------------------------

Total listings after filtering: 99
Median Quality Score: 8.0

Listings count per score:
quality_score
10    18
9     12
8     34
7      2
6     18
5      7
4      7
3      1
Name: count, dtype: int64

Bottom 10 listings by score:


,title,price,quality_score
55,2-bedroom furnished flat for rent in...,1480,3
58,3-bedroom furnished apartment with h...,2400,4
38,Flat with heating in Raval Barcelona,174000,4
59,2-bedroom house for rent with outdoo...,1790,4
56,2-bedroom apartment with outdoor are...,2100,4
82,2-bedroom furnished flat with terrac...,1890,4
94,Furnished 2-bedroom flat for rent in...,1250,4
91,2-bedroom furnished flat for rent in...,1890,4
72,3-bedroom furnished flat with air co...,2500,5
69,2-bedroom furnished flat for rent in...,1750,5


## 7. Vector Embeddings
In this step, we convert the property descriptions into numerical vectors (embeddings). These embeddings allow us to perform semantic searches and find similar properties based on their content rather than just keywords.

In [83]:
import os
import numpy as np

def ensure_results_loaded(file_path):
    """Loads results from JSON if the variable is missing or empty."""
    global results, df
    if 'results' not in globals() or not results:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                results = json.load(f)
            print(f"Loaded {len(results)} items from {file_path}.")
        else:
            print("No results found in memory or local file.")
            return False
    return True

def generate_embeddings(items, model="text-embedding-3-small"):
    """Generates embeddings for descriptions in batches."""
    texts = [str(item.get('description', 'No description')).replace('\n', ' ')[:8000] for item in items]

    print(f"Generating embeddings for {len(texts)} items...")
    try:
        response = client.embeddings.create(input=texts, model=model)
        embeddings = [data.embedding for data in response.data]

        # Map embeddings back to results and update DataFrame
        for i, item in enumerate(items):
            item['embedding'] = embeddings[i]
        return embeddings
    except Exception as e:
        print(f"Error generating embeddings: {e}")
        return None

# Execution
if ensure_results_loaded(OUTPUT_FILE):
    embeddings = generate_embeddings(results)
    if embeddings:
        df['embedding'] = embeddings
        print("Embeddings successfully attached to results and DataFrame.")

Generating embeddings for 99 items...
Embeddings successfully attached to results and DataFrame.


## 7. Export & Preview
Finally, we save the structured data to a JSON file and provide a preview of the results using a Pandas DataFrame.

In [84]:
# ── 6. SAVE TO JSON ───────────────────────────────────────────────────────────


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} listings → {OUTPUT_FILE}")

Saved 99 listings → habitaclia_listings.json


In [85]:
# ── 7. QUICK PREVIEW ─────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)

PREVIEW_COLS = [
    "title", "transaction_type", "listing_type",
    "price", "surface_m2", "rooms", "bathrooms",
    "neighborhood", "city",
]
cols = [c for c in PREVIEW_COLS if c in df.columns]
pd.set_option("display.max_colwidth", 40)
df[cols]

,title,transaction_type,listing_type,price,surface_m2,rooms,bathrooms,neighborhood,city
0,Flat in La Nova Esquerra de l'Eixamp...,buy,flat,499000,75,2,1.0,La Nova Esquerra de l'Eixample,Barcelona
1,Flat with heating in Dreta de l'Eixa...,buy,flat,1175000,150,3,3.0,Dreta de l'Eixample,Barcelona
2,Penthouse with heating in L’Antiga E...,buy,penthouse,1250000,129,3,2.0,L’Antiga Esquerra de l’Eixample,Barcelona
3,Renovated 4-bedroom flat near Sagrad...,buy,flat,538000,92,4,2.0,Sagrada Família,Barcelona
4,Flat with heating in Sagrada Família...,buy,flat,435000,65,2,1.0,Sagrada Família,Barcelona
...,...,...,...,...,...,...,...,...,...
94,Furnished 2-bedroom flat for rent in...,rent,flat,1250,30,2,1.0,Barceloneta,Barcelona
95,2-bedroom furnished flat with air co...,rent,flat,2322,1,2,1.0,Dreta de l'Eixample,Barcelona
96,Modern furnished flat for seasonal r...,rent,flat,2500,64,2,2.0,Dreta de l'Eixample,Barcelona
97,Unique duplex oasis with private gar...,rent,duplex,8000,625,4,2.0,Sant Antoni,Barcelona


In [86]:
# ── 8. DOWNLOAD (Google Colab only) ──────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>